# Arquitetura em Camadas (Layers) — Exemplo Executável

**Guia Aberto de Arquitetura de Software** · Tópico: Arquitetura em Camadas

**Autores:**
- Maria Letícia de Sousa Barboza
- João Henrique Lopes de Araújo Freire
- Pedro Henrique Reis Xavier
- Wellison Danniel Soares Ponciano
- Caio Vinícius Santana Gomes
- Ruan Henrique Pereira dos Santos

**Licença:** CC BY-NC 4.0

---


Este notebook é o exemplo do tópico. Implementa um sistema de
**gerenciamento de tarefas** seguindo o modelo de três camadas:

- **Persistência** — acesso aos dados (simulado em memória)
- **Domínio (Lógica de Negócio)** — regras, validações
- **Apresentação** — recebe entrada do usuário e exibe resultados

Regra de dependência: **Apresentação → Domínio → Persistência** (nunca o inverso).

Estrutura desta demonstração:

1. **Roda de ponta a ponta** em ambiente limpo (sem dependências externas)
2. **Demonstra o conceito**, não apenas ilustra
3. **Mostra a violação** da restrição arquitetural e sua consequência
4. **Explica em células de texto**, não só em comentários de código

## 0. Dependências

Sem bibliotecas externas — só a biblioteca padrão do Python.

In [ ]:
import sys
import dis

print(f"Python {sys.version.split()[0]}")

## 1. O problema

Um sistema de tarefas precisa: validar dados de entrada (título não pode ser
vazio, prioridade precisa ser uma das opções válidas), guardar os dados em
algum lugar, e expor isso para quem usa o sistema — seja um terminal, seja uma
API.

Se essas três preocupações forem misturadas na mesma classe, qualquer mudança
em uma (trocar de memória para banco de dados, adicionar uma nova regra de
validação, trocar CLI por API web) arrisca quebrar as outras duas. O notebook
abaixo implementa o sistema separando essas responsabilidades em camadas e, na
seção 3, mostra concretamente o que acontece quando essa separação é violada.

## 2. Implementação seguindo o estilo

### 2.1 Camada de Persistência (Data Access Layer)

Responsável apenas por armazenar e recuperar dados. Não conhece regras de
negócio nem interface — não valida nada, só guarda o que recebe.

In [ ]:
class TarefaRepository:
    """Simula um banco de dados em memória.
    Responsabilidade única: salvar, ler, atualizar e remover registros.
    Não decide se um dado é válido — isso é papel do Domínio."""

    def __init__(self):
        self._tarefas = {}
        self._proximo_id = 1

    def salvar(self, dados: dict) -> int:
        tarefa_id = self._proximo_id
        self._tarefas[tarefa_id] = dados
        self._proximo_id += 1
        return tarefa_id

    def buscar_todas(self) -> list:
        return [dict(t, id=tid) for tid, t in self._tarefas.items()]

    def buscar_por_id(self, tarefa_id: int) -> dict | None:
        tarefa = self._tarefas.get(tarefa_id)
        return dict(tarefa, id=tarefa_id) if tarefa else None

    def atualizar(self, tarefa_id: int, dados: dict) -> bool:
        if tarefa_id not in self._tarefas:
            return False
        self._tarefas[tarefa_id] = dados
        return True

    def remover(self, tarefa_id: int) -> bool:
        return self._tarefas.pop(tarefa_id, None) is not None

### 2.2 Camada de Domínio (Business Logic Layer)

O "coração" do sistema. Aplica validações — título obrigatório e prioridade
dentro de um conjunto válido. Depende da Persistência, mas a Persistência
**não** conhece o Domínio.

In [ ]:
class TarefaInvalidaError(Exception):
    pass


class TarefaService:
    """Contém as regras de negócio. É a única camada que decide o que é uma
    tarefa válida e o que acontece quando ela é criada, concluída, etc."""

    PRIORIDADES_VALIDAS = {"baixa", "media", "alta"}

    def __init__(self, repositorio: TarefaRepository):
        self._repo = repositorio

    def criar_tarefa(self, titulo: str, prioridade: str = "media") -> dict:
        if not titulo or not titulo.strip():
            raise TarefaInvalidaError("O título da tarefa não pode ser vazio.")
        if prioridade not in self.PRIORIDADES_VALIDAS:
            raise TarefaInvalidaError(
                f"Prioridade inválida: {prioridade!r}. Use uma de {self.PRIORIDADES_VALIDAS}."
            )

        dados = {"titulo": titulo.strip(), "prioridade": prioridade, "concluida": False}
        tarefa_id = self._repo.salvar(dados)
        return self._repo.buscar_por_id(tarefa_id)

    def concluir_tarefa(self, tarefa_id: int) -> dict:
        tarefa = self._repo.buscar_por_id(tarefa_id)
        if tarefa is None:
            raise TarefaInvalidaError(f"Tarefa {tarefa_id} não existe.")
        tarefa["concluida"] = True
        self._repo.atualizar(tarefa_id, {k: v for k, v in tarefa.items() if k != "id"})
        return tarefa

    def listar_pendentes(self) -> list:
        return [t for t in self._repo.buscar_todas() if not t["concluida"]]

    def listar_todas_ordenadas(self) -> list:
        ordem = {"alta": 0, "media": 1, "baixa": 2}
        return sorted(self._repo.buscar_todas(), key=lambda t: ordem[t["prioridade"]])

### 2.3 Camada de Apresentação (Presentation Layer)

Só traduz a interação do usuário em chamadas ao Domínio, e formata a saída.
Não contém nenhuma regra de negócio — repare que ela nunca acessa
`TarefaRepository` diretamente, apenas `TarefaService`.

In [ ]:
class PresentationCLI:
    """Uma 'interface' simples baseada em funções, simulando um terminal.
    Poderia ser trocada por uma interface web sem tocar no Domínio."""

    def __init__(self, service: TarefaService):
        self._service = service

    def adicionar(self, titulo: str, prioridade: str = "media"):
        try:
            tarefa = self._service.criar_tarefa(titulo, prioridade)
            print(f"Tarefa criada: #{tarefa['id']} — {tarefa['titulo']} [{tarefa['prioridade']}]")
        except TarefaInvalidaError as erro:
            print(f"Erro: {erro}")

    def concluir(self, tarefa_id: int):
        try:
            tarefa = self._service.concluir_tarefa(tarefa_id)
            print(f"Tarefa #{tarefa_id} marcada como concluída.")
        except TarefaInvalidaError as erro:
            print(f"Erro: {erro}")

    def exibir_pendentes(self):
        pendentes = self._service.listar_pendentes()
        print("--- Tarefas pendentes ---")
        if not pendentes:
            print("(nenhuma)")
        for t in pendentes:
            print(f"#{t['id']} [{t['prioridade']}] {t['titulo']}")


repositorio = TarefaRepository()
servico = TarefaService(repositorio)
tela = PresentationCLI(servico)

tela.adicionar("Estudar arquitetura em camadas", "alta")
tela.adicionar("Revisar slides de HCI", "media")
tela.adicionar("", "alta")            # dispara erro de validação (título vazio)
tela.adicionar("Tarefa X", "urgente") # dispara erro de validação (prioridade inválida)
print()
tela.exibir_pendentes()

### Por que isso é testável

A consequência prática de respeitar a restrição: o domínio pode ser exercitado
sem persistência real, porque `TarefaService` depende só da interface pública
de `TarefaRepository` (métodos `salvar`/`buscar_por_id`), não de um banco de
dados de verdade.

In [ ]:
class RepositorioFalso(TarefaRepository):
    """Dublê de teste — mesma interface pública, sem nenhuma infraestrutura
    real por trás. Possível justamente porque o Domínio depende da interface,
    não de uma implementação específica."""
    pass


falso = RepositorioFalso()
TarefaService(falso).criar_tarefa("Testar sem banco de dados", "alta")

assert len(falso.buscar_todas()) == 1
print("Teste passou — domínio exercitado sem banco de dados real.")

## 3. A violação

Agora mostramos o que acontece quando a restrição do estilo é quebrada. Esta
seção é obrigatória: é ela que transforma o notebook em demonstração, e não
em ilustração.

Abaixo, uma segunda "apresentação" — um gerador de relatório — que, por
pressa, acessa `TarefaRepository` **diretamente**, pulando `TarefaService`
para "economizar uma chamada". O problema: as validações de negócio (título
obrigatório, prioridade válida) vivem só no `TarefaService`, então pular essa
camada significa pular as validações.

In [ ]:
# ANTIPADRÃO: a Apresentação conhece a Persistência diretamente.
# A dependência agora aponta Apresentação -> Persistência, pulando o Domínio,
# e a regra de arquitetura (Apresentação -> Domínio -> Persistência) foi violada.

class RelatorioDireto:
    """Gera um relatório rápido inserindo tarefas 'de sistema' direto no
    repositório, sem passar pelo TarefaService."""

    def __init__(self, repositorio: TarefaRepository):
        self._repo = repositorio  # aponta para a Persistência, não para o Domínio

    def registrar_tarefa_de_sistema(self, titulo: str, prioridade: str):
        # Nenhuma validação acontece aqui — TarefaInvalidaError nunca é levantada,
        # porque essa classe nunca passa pelo TarefaService.
        dados = {"titulo": titulo, "prioridade": prioridade, "concluida": False}
        return self._repo.salvar(dados)


relatorio = RelatorioDireto(repositorio)
id_invalida = relatorio.registrar_tarefa_de_sistema("", "urgentissimo")

tarefa_salva = repositorio.buscar_por_id(id_invalida)
print(f"Tarefa #{id_invalida} salva com título vazio e prioridade inválida: {tarefa_salva}")
print("Consequência: dado que o TarefaService jamais permitiria existe no banco.")

## 4. Consequência mensurável

Em vez de só afirmar que RelatorioDireto "viola o estilo", medimos isso de forma verificável: pegamos o código-fonte de cada método das classes de Apresentação e contamos quantas vezes citam, por nome, a classe concreta de Persistência (TarefaRepository). Quem respeita a arquitetura só deveria referenciar TarefaService.

In [ ]:
import inspect

INFRA = "TarefaRepository"

def referencias_persistencia(cls) -> int:
    """Soma, no código-fonte de cada método da classe, quantas vezes o nome
    da classe concreta de Persistência aparece."""
    total = 0
    for nome, membro in vars(cls).items():
        try:
            codigo_fonte = inspect.getsource(membro)
        except (TypeError, OSError):
            continue
        total += codigo_fonte.count(INFRA)
    return total


for cls in (PresentationCLI, RelatorioDireto):
    n = referencias_persistencia(cls)
    situacao = "viola o estilo" if n else "respeita o estilo"
    print(f"{cls.__name__:16} referencias a {INFRA}: {n} -> {situacao}")

## 5. Conclusão

`PresentationCLI` só conhece `TarefaService` — trocar a persistência ou mudar
a interface não exige tocar nas regras de negócio. `RelatorioDireto` mostra o
outro lado do trade-off: pular uma camada para "ganhar tempo" custa a
garantia de que os dados salvos são válidos, e esse custo fica invisível até
alguém consultar o banco diretamente.

---

### Referências

[1] F. Buschmann, R. Meunier, H. Rohnert, P. Sommerlad e M. Stal, *Pattern-Oriented Software Architecture: A System of Patterns*. John Wiley & Sons, 1996.

[2] L. Bass, P. Clements e R. Kazman, *Software Architecture in Practice*, 2ª ed. Addison-Wesley, 2003.

[3] I. Sommerville, *Engenharia de Software*, 9ª ed. Pearson, 2011.

[4] M. Fowler, "PresentationDomainDataLayering", *martinfowler.com*, ago. 2015. Disponível em: https://martinfowler.com/bliki/PresentationDomainDataLayering.html

---

Conteúdo sob CC BY-NC 4.0 — uso livre com crédito. Código sob MIT.